In [5]:
import pandas as pd 
conversion_data = pd.read_csv(r'dataset\conversations.csv', header=None)
conversion_data.head()

,0
0,"User 1: Hi! How are you?\nUser 2: Good, thanks..."
1,User 1: Hey how are you doing?\nUser 2: Doing ...
2,"User 1: Hello there, how are you doing today?\..."
3,User 1: hi there!\nUser 2: Hey! How's your day...
4,User 1: How is your day going?\nUser 2: Great!...


Topic Checkpoints

In [ ]:
import os
import json
import torch
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from keybert import KeyBERT

# --- 1. INITIALIZE LOCAL MODELS ---
print("Loading Embedding Model (MiniLM)...")
# Automatically uses GPU if available, otherwise falls back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print(f"Loading Summarization Model (BART) on {device}...")

kw_model = KeyBERT(model=embedder) 

# Use the dialogue-specific summarizer
summarizer = pipeline("summarization", model="philschmid/bart-large-cnn-samsum", device=0 if device == "cuda" else -1)

# --- 2. CORE LOGIC: SEMANTIC TEXTTILING ---
def process_conversation(conv_text, conv_id, threshold=0.35):
    messages = [msg.strip() for msg in conv_text.split('\n') if msg.strip()]
    embeddings = embedder.encode(messages, convert_to_tensor=True)
    
    checkpoints = []
    current_segment = [messages[0]]
    start_idx = 1
    
    for i in range(1, len(messages)):
        sim = util.cos_sim(embeddings[i], embeddings[i-1]).item()
        
        if sim < threshold:
            segment_text = " ".join(current_segment)
            input_length = len(segment_text.split())
            max_len = min(60, max(20, input_length))
            min_len = min(20, max(5, input_length // 2))
            
            # --- 1. GENERATE ABSTRACTIVE SUMMARY ---
            try:
                summary_obj = summarizer(segment_text, max_length=max_len, min_length=min_len, do_sample=False)[0]
                summary_text = summary_obj['summary_text']
            except Exception as e:
                summary_text = f"Could not summarize: {str(e)}"
            
            # --- 2. GENERATE INTELLIGENT TOPIC LABEL ---
            # Extract the top 1 phrase that is 1 to 3 words long
            try:
                keywords = kw_model.extract_keywords(
                    segment_text, 
                    keyphrase_ngram_range=(1, 3), 
                    stop_words='english', 
                    top_n=1
                )
                # Fallback to "General Discussion" if KeyBERT finds nothing
                topic_label = keywords[0][0].title() if keywords else "General Discussion"
            except:
                topic_label = "General Discussion"

            checkpoints.append({
                "topic_id": len(checkpoints) + 1,
                "range": f"{start_idx}-{i}",
                "topic_label": topic_label,
                "summary": summary_text
            })
            
            start_idx = i + 1
            current_segment = [messages[i]]
        else:
            current_segment.append(messages[i])

    # Handle the final remaining segment
    if current_segment:
        segment_text = " ".join(current_segment)
        input_length = len(segment_text.split())
        max_len = min(60, max(20, input_length))
        min_len = min(20, max(5, input_length // 2))
        
        try:
            summary_obj = summarizer(segment_text, max_length=max_len, min_length=min_len, do_sample=False)[0]
            summary_text = summary_obj['summary_text']
        except Exception as e:
            summary_text = f"Could not summarize: {str(e)}"
            
        checkpoints.append({
            "topic_id": len(checkpoints) + 1,
            "range": f"{start_idx}-{len(messages)}",
            "topic_label": summary_text[:40] + "...",
            "summary": summary_text
        })

    return {"conversation_id": conv_id, "checkpoints": checkpoints}

# --- 3. EXECUTION: CHUNKED SAVING ---
def process_all_conversations_chunked(input_csv_path, output_json_path, chunk_size=50):
    print(f"Loading dataset from: {input_csv_path}")
    df = pd.read_csv(input_csv_path, header=None)
    
    all_results = []
    processed_ids = set()
    
    # Resume functionality
    if os.path.exists(output_json_path):
        try:
            with open(output_json_path, 'r', encoding='utf-8') as f:
                all_results = json.load(f)
                processed_ids = {res["conversation_id"] for res in all_results if "error" not in res}
            print(f"Found existing JSON. Resuming... {len(processed_ids)} conversations already processed.")
        except json.JSONDecodeError:
            print("Existing JSON is corrupted or empty. Starting fresh.")

    records_to_process = []
    for index, row in df.iterrows():
        conv_id = index + 1 
        if conv_id not in processed_ids:
            records_to_process.append((conv_id, row[0]))

    if not records_to_process:
        print("All conversations in the dataset have already been processed!")
        return

    print(f"{len(records_to_process)} remaining conversations to process.")

    # Process sequentially in chunks
    for i in range(0, len(records_to_process), chunk_size):
        chunk = records_to_process[i:i+chunk_size]
        start_id = chunk[0][0]
        end_id = chunk[-1][0]
        print(f"\n--- Processing Batch: IDs {start_id} to {end_id} ({len(chunk)} records) ---")

        chunk_results = []
        
        # Sequential processing (PyTorch handles hardware optimization)
        for conv_id, text in tqdm(chunk, desc="Batch Progress"):
            try:
                result = process_conversation(text, conv_id)
                chunk_results.append(result)
            except Exception as e:
                print(f"\nError processing conversation {conv_id}: {e}")
                chunk_results.append({"conversation_id": conv_id, "error": str(e), "checkpoints": []})

        all_results.extend(chunk_results)

        # Save to disk
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(all_results, f, indent=2, ensure_ascii=False)
            
        print(f"Batch saved successfully. Total processed so far: {len(all_results)}")

if __name__ == "__main__":
    input_file = r'/kaggle/input/datasets/saitharshith/conversation-dataset/conversations.csv'
    output_file = r'/kaggle/working/Topic_checkpoints.json'
    
    output_dir = os.path.dirname(output_file)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        
    process_all_conversations_chunked(
        input_file, 
        output_file, 
        chunk_size=50 
    )

100 Message Checkpoints

In [ ]:
import os
import json
import pandas as pd
from tqdm import tqdm
import torch
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Hugging Face SAMSum model on {device}...")

summarizer = pipeline(
    "summarization", 
    model="philschmid/bart-large-cnn-samsum", 
    device=0 if device == "cuda" else -1
)

def create_global_100_msg_checkpoints(input_csv_path, output_json_path):
    print("Loading dataset...")
    df = pd.read_csv(input_csv_path, header=None)
    
    # 1. FLATTEN THE DATASET INTO A CONTINUOUS STREAM
    print("Flattening dataset into a global message stream...")
    global_stream = []
    
    for index, row in df.iterrows():
        conv_id = index + 1
        # Split row into individual messages
        messages = [msg.strip() for msg in row[0].split('\n') if msg.strip()]
        
        # Tag every single message with its original location
        for msg_idx, msg in enumerate(messages):
            global_stream.append({
                "conv_id": conv_id,
                "msg_text": msg,
                "local_idx": msg_idx + 1
            })

    print(f"Total individual messages extracted: {len(global_stream)}")
    
    checkpoints = []
    
    # 2. SLICE INTO STRICT 100-MESSAGE CHUNKS
    total_chunks = len(global_stream) // 100
    
    for i in tqdm(range(0, len(global_stream), 100), total=total_chunks, desc="Generating 100-Msg Summaries"):
        chunk = global_stream[i:i+100]
        
        # Skip the final chunk if it's just a tiny remnant of messages
        if len(chunk) < 10: 
            continue
            
        chunk_text = "\n".join([item["msg_text"] for item in chunk])
        
        # Metadata Tracking: Find exactly where this chunk starts and ends
        start_conv = chunk[0]["conv_id"]
        end_conv = chunk[-1]["conv_id"]
        start_msg = chunk[0]["local_idx"]
        end_msg = chunk[-1]["local_idx"]
        
        # Extract all unique conversation IDs involved in this 100-message block
        conv_ids_involved = list(set([item["conv_id"] for item in chunk]))
        
        # Dynamic summarizer constraints
        input_length = len(chunk_text.split())
        max_len = min(100, max(30, input_length))
        min_len = min(30, max(10, input_length // 2))
        
        try:
            result = summarizer(
                chunk_text, 
                max_length=max_len,  
                min_length=min_len,   
                do_sample=False, 
                truncation=True 
            )
            summary = result[0]['summary_text']
        except Exception as e:
            summary = f"Error generating summary: {str(e)}"
            
        checkpoints.append({
            "global_checkpoint_id": (i // 100) + 1,
            "span_start": f"Conv {start_conv}, Msg {start_msg}",
            "span_end": f"Conv {end_conv}, Msg {end_msg}",
            "conversations_included": conv_ids_involved,
            "summary": summary
        })

        # Auto-save every 50 checkpoints
        if len(checkpoints) % 50 == 0:
            with open(output_json_path, 'w', encoding='utf-8') as f:
                json.dump(checkpoints, f, indent=2, ensure_ascii=False)

    # Final save
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(checkpoints, f, indent=2, ensure_ascii=False)
        
    print(f"\nSaved {len(checkpoints)} strict 100-message checkpoints to {output_json_path}")

if __name__ == "__main__":
    input_file = r'/kaggle/input/datasets/saitharshith/conversation-dataset/conversations.csv'
    output_file = r'/kaggle/working/100_interval_checkpoints.json'
    
    output_dir = os.path.dirname(output_file)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        
    create_global_100_msg_checkpoints(input_file, output_file)

Vector Database 

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from tqdm import tqdm

def build_vector_database(topic_json_path, interval_json_path, db_path):
    print("Initializing ChromaDB...")
    client = chromadb.PersistentClient(path=db_path)
    sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
    collection = client.get_or_create_collection(
        name="conversation_checkpoints",
        embedding_function=sentence_transformer_ef
    )
    
    documents = []
    metadatas = []
    ids = []

    # --- 4. LOAD TOPIC CHECKPOINTS ---
    if os.path.exists(topic_json_path):
        print("Loading Topic Checkpoints...")
        with open(topic_json_path, 'r', encoding='utf-8') as f:
            topic_data = json.load(f)
            
        for conv in topic_data:
            conv_id = conv.get("conversation_id", "Unknown")
            for checkpoint in conv.get("checkpoints", []):
                # The text that actually gets vectorized
                documents.append(checkpoint["summary"])
                # The metadata attached to the vector
                metadatas.append({
                    "source_type": "topic_checkpoint",
                    "conversation_id": str(conv_id),
                    "topic_label": checkpoint.get("topic_label", "Unknown"),
                    "message_range": checkpoint.get("range", "Unknown")
                })
                
                # A unique ID for the database
                ids.append(f"topic_c{conv_id}_{checkpoint['topic_id']}")

    # --- 5. LOAD INTERVAL CHECKPOINTS ---
    if os.path.exists(interval_json_path):
        print("Loading Interval Checkpoints...")
        with open(interval_json_path, 'r', encoding='utf-8') as f:
            interval_data = json.load(f)
            
        for chunk in interval_data:
            documents.append(chunk["summary"])
            
            metadatas.append({
                "source_type": "interval_checkpoint",
                "conversations_included": str(chunk.get("conversations_included", [])),
                "span_start": chunk.get("span_start", "Unknown"),
                "span_end": chunk.get("span_end", "Unknown")
            })
            
            ids.append(f"interval_global_{chunk['global_checkpoint_id']}")

   
    print(f"Total documents to index: {len(documents)}")
    print("Embedding and saving to database (this may take a moment)...")
    batch_size = 5000
    for i in tqdm(range(0, len(documents), batch_size), desc="Upserting to ChromaDB"):
        batch_docs = documents[i:i+batch_size]
        batch_meta = metadatas[i:i+batch_size]
        batch_ids = ids[i:i+batch_size]
        
        collection.upsert(
            documents=batch_docs,
            metadatas=batch_meta,
            ids=batch_ids
        )
        
    print(f"\nVector database successfully built and saved to: {db_path}")
    return collection


if __name__ == "__main__":
    TOPIC_FILE = r'/kaggle/input/datasets/saitharshith/checkpoints/Topic_checkpoints.json'
    INTERVAL_FILE = r'/kaggle/input/datasets/saitharshith/checkpoints/interval_checkpoints.json'
    DB_DIRECTORY = r'/kaggle/working/rag_chroma_db'
    db_collection = build_vector_database(TOPIC_FILE, INTERVAL_FILE, DB_DIRECTORY)

User Persona

In [ ]:
import json
import torch
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Loading {MODEL_ID} into GPU...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16 
)
def extract_dual_personas(text_chunk):
    system_prompt = """You are an expert behavioral analyst and a strict data extraction algorithm.
Your ONLY objective is to analyze conversation summaries involving two distinct speakers (User 1 and User 2) and extract their behavioral profiles. You must output the result EXCLUSIVELY as a valid JSON object.

### CATEGORY DEFINITIONS
- "habits": Recurring actions, daily routines, or lifestyle choices (e.g., "wakes up early", "plays guitar daily").
- "personal_facts": Objective life details, relationships, jobs, locations, or past events (e.g., "moving to Portland", "is a nurse", "owns a dog").
- "personality_traits": Inherent characteristics or adjectives describing their demeanor (e.g., "optimistic", "anxious", "humorous").
- "communication_style": How they interact, including tone, sentence structure, or emotional expression (e.g., "asks many questions", "uses brief sentences", "highly enthusiastic").

### STRICT RULES
1. ZERO HALLUCINATION: Base your extraction STRICTLY on the provided text. Do not infer or guess.
2. ATTRIBUTION: Distinguish carefully between User 1 and User 2. Attribute facts and traits only to the correct speaker based on the context.
3. EMPTY ARRAYS: If no evidence exists for a specific category for a user, you MUST leave the array empty []. Do not write "None" or "N/A".
4. CONCISENESS: Keep each extracted point brief (maximum 8 words per item).
5. STRICT JSON FORMAT: Output ONLY the raw JSON object. DO NOT wrap the JSON in markdown code blocks (e.g., do not use ```json). DO NOT output any conversational text, greetings, or explanations before or after the JSON.

### EXPECTED OUTPUT SCHEMA
{
  "User 1": {
    "habits": [],
    "personal_facts": [],
    "personality_traits": [],
    "communication_style": []
  },
  "User 2": {
    "habits": [],
    "personal_facts": [],
    "personality_traits": [],
    "communication_style": []
  }
}"""

    user_prompt = f"Analyze this text and extract the dual JSON profile:\n{text_chunk}\n\nOUTPUT STRICT JSON ONLY:"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=350, 
            temperature=0.1, 
            do_sample=False
        )
        
    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    
    try:
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return None
    except json.JSONDecodeError:
        return None


def build_per_conversation_personas(topic_json_path, output_json_path, sample_limit=50):
    print("Loading Topic Checkpoints...")
    with open(topic_json_path, 'r', encoding='utf-8') as f:
        topic_data = json.load(f)
        
    conversation_personas_db = {}
    
    conversations_to_process = topic_data[:sample_limit]
    print(f"Extracting traits from {len(conversations_to_process)} distinct conversations...")

    for conv in tqdm(conversations_to_process, desc="Analyzing Personas per Conversation"):
        conv_id = conv.get("conversation_id")
        conv_summaries = [cp["summary"] for cp in conv.get("checkpoints", [])]
        chunk_text = "\n".join(conv_summaries)
        extracted_data = extract_dual_personas(chunk_text)
        
        if extracted_data:
            conversation_personas_db[f"conversation_{conv_id}"] = extracted_data

    # Save the database to disk
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(conversation_personas_db, f, indent=2)
        
    print(f"\nPer-Conversation Personas successfully extracted and saved to: {output_json_path}")
    return conversation_personas_db

# --- EXECUTE ---
if __name__ == "__main__":
    TOPIC_FILE = r'/kaggle/input/datasets/saitharshith/checkpoints/Topic_checkpoints.json'
    PERSONA_FILE = r'/kaggle/working/per_conversation_personas.json'
    profile_db = build_per_conversation_personas(TOPIC_FILE, PERSONA_FILE, sample_limit=50)

Chatbot

In [ ]:
import os
import json
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv


load_dotenv()

llm = ChatGroq(
    model_name="llama-3.1-8b-instant", 
    temperature=0.1,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = Chroma(
    persist_directory="chroma_database", 
    embedding_function=embeddings
)
with open(r'dataset/dual_user_personas.json', 'r', encoding='utf-8') as f:
    persona_db = json.load(f)

def run_behavioral_bot(question, conv_id, target_user="User 1"):

    conv_key = f"conversation_{conv_id}"
    persona_data = persona_db.get(conv_key, {}).get(target_user, "No profile data available.")
    persona_context = json.dumps(persona_data, indent=2)
    
    # B. Retrieve Vector Context
    retriever = vector_db.as_retriever(
        search_kwargs={"k": 3, "filter": {"conversation_id": str(conv_id)}}
    )
    docs = retriever.invoke(question)
    retrieved_context = "\n\n".join([doc.page_content for doc in docs])
    
    # C. Define the Prompt Structure
    system_message = """### ROLE
You are an Expert Behavioral Analyst. Your objective is to deliver a definitive, evidence-based psychological profile of the requested user. 

### CONTEXT DATA
You have access to two distinct data sources:
1. [PERSONA PROFILE]: Hard facts and extracted traits.
2. [CONVERSATION SUMMARIES]: Contextual evidence of their behavior in action.

### STRICT RULES OF ANALYSIS
1. NO META-CHATTER: Do not reference the system prompt, the "JSON", or the "provided summaries" in your output. Speak directly about the user. (e.g., Instead of saying "The JSON indicates they read," say "User 1 is an avid reader.")
2. ZERO SPECULATION: Do not guess. If a trait (like communication style) is missing, state: "There is insufficient data to determine their communication style." DO NOT attempt to guess their communication style based on their habits.
3. NO FOLLOW-UPS: You are writing a final report. Do not ask the user for more information or offer to revise your analysis. 
4. SYNTHESIS: Seamlessly weave the facts and the conversation summaries together. Show *how* their traits appear in their dialogue.

### REQUIRED OUTPUT FORMAT
Format your response using professional markdown. Use these exact headers:
**Behavioral Profile: {target_user}**
*Write a short 2-sentence executive summary here.*

**Established Habits & Facts**
*Detail their lifestyle based on the data.*

**Communication & Personality**
*Detail how they speak and act. If data is missing, state it firmly and move on.*
"""

    human_message = """### INPUT DATA
User being analyzed: {target_user}

[PERSONA PROFILE]: 
{persona_context}

[CONVERSATION SUMMARIES]: 
{retrieved_context}

### USER QUERY
{question}

### RESPONSE
"""

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_message),
        ("human", human_message)
    ])

    # D. Execute Chain
    chain = prompt | llm | StrOutputParser()
    
    response = chain.invoke({
        "target_user": target_user,
        "persona_context": persona_context,
        "retrieved_context": retrieved_context,
        "question": question
    })
    
    return response

if __name__ == "__main__":
    test_conv = "3" 
    test_query = "What are this user's habits and how do they talk?"
    result = run_behavioral_bot(test_query, conv_id=test_conv, target_user="User 1")
    print(f"\n--- Chatbot Response (Conv {test_conv}) ---\n")
    print(result)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 857.58it/s]



--- Chatbot Response (Conv 3) ---

**Behavioral Profile: User 1**
User 1 is a well-rounded individual with a strong sense of family and a passion for self-sufficiency. Their habits and communication style reveal a warm and engaging personality.

**Established Habits & Facts**
User 1 has a variety of established habits that showcase their commitment to personal growth and relationships. They are an avid reader, often spending time with books, which suggests a curious and introspective nature. Additionally, their love for cooking indicates a creative and nurturing side. Spending time with family is also a priority, highlighting their importance on close relationships. Furthermore, their ownership of a dog and a cat demonstrates a compassionate and empathetic personality.

**Communication & Personality**
There is insufficient data to determine their communication style. However, based on their habits, it can be inferred that User 1 is likely a warm and engaging conversationalist. Their p